# Presentation of our solution -- Quantum urban vision

Today's world is obeserved through the lens of multiple devices. Here, we consider images of our earth took from satellites. All sort of things can be spotted through these images. The idea is to train a quantum model to flag some target components that we could find on these images. Here, for simplicity, we will consider a narrower dataset. 

We assume that our model can only input 10 type of images, all with the same dimensions: planes, airports, ships, harbor, tennis court, basketball court, clouds, dense residential, railway station, river. Our goal is to follow the traffic both in the air and in the water. That means that our model should flag us when it sees any airport, ship, plane, or harbor. 

Now, knowing an image has either a ship or a plane is already good, but not sufficient, since these two are broadly different. We can thus train another model that takes in entry an image of either a plane, a boat, a harbor, or an airport, and that classifies the image in its corresponding class. 

To train our models, we used an already existing dataset called `NWPU-RESISC45`. It contains images from satellites classified in 45 classes. We cutted all classes but the previously mentionned classes of interest, and use them as training data and testing data. 

## A sneak peak of our classification model


### 1. Training a QSVM algorithm to flag or not the image as a targeted image.

The first step of our pipeline is to train our QSVM model to actually recognise the targeted images. Here, we want to recognise if the image is either a plane, a boat, an airport, or a harbor. Then, we apply this trained model on our test data to highlight the potential anomalies.

In [1]:

from anomaly_detector import create_and_train_classifier, create_anomaly_dataset, extract_image_features, feature_reductor, create_quantum_kernels, detect_anomaly
from sklearn.metrics import roc_auc_score
import numpy as np

import torch
import random

#Fixing the seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

# ====================================================
# DATA
# ====================================================

train_tensor = create_anomaly_dataset(0.0, 100)

x_train = train_tensor[0][0]
y_train = train_tensor[0][1]

test_tensor = create_anomaly_dataset(0.1, 100)

x_test = test_tensor[1][0]
y_test = test_tensor[1][1]


# ====================================================
# FEATURE EXTRACTION
# ====================================================

train_features = extract_image_features(x_train)
test_features = extract_image_features(x_train)

print("Train features:", train_features.shape)
print("Test features:", test_features.shape)


# ====================================================
# PCA + scaler -> 8 DIMENSIONS
# ====================================================

train_pca = feature_reductor(train_features, num_features=8)
test_pca = feature_reductor(train_features, num_features=8)


# ====================================================
# QUANTUM KERNEL
# ====================================================

train_kernel, test_kernel = create_quantum_kernels(x_train=train_pca, x_test=test_pca, num_features=8)


# ====================================================
# ONE-CLASS SVM
# ====================================================
# Unlike standard SVMs that require data from two classes to find a separating boundary, a One-Class SVM trains only on "normal" data to learn its distribution, flagging anything that deviates significantly from this norm
ocsvm = create_and_train_classifier(train_kernel)


# ====================================================
# ANOMALY SCORES
# ====================================================

decision_scores, preds = detect_anomaly(ocsvm, test_kernel)


# ====================================================
# CONFUSION MATRIX COMPONENTS
# ====================================================

y_test=y_test.numpy()  # Convert to NumPy array if it's a PyTorch tensor

TP = np.sum((preds == 1) & (y_test == 1))
FP = np.sum((preds == 1) & (y_test == 0))
TN = np.sum((preds == 0) & (y_test == 0))
FN = np.sum((preds == 0) & (y_test == 1))

print("\n==============================")
print("CONFUSION MATRIX SUMMARY")
print("==============================")
print(f"True Positives  (TP): {TP}")
print(f"False Positives (FP): {FP}")
print(f"True Negatives  (TN): {TN}")
print(f"False Negatives (FN): {FN}")

Train features: (100, 512)
Test features: (100, 512)
Building training kernel matrix...
Building test kernel matrix...

CONFUSION MATRIX SUMMARY
True Positives  (TP): 0
False Positives (FP): 10
True Negatives  (TN): 80
False Negatives (FN): 10


Second, we can see how our model does when it comes to classify between 4 images. 

In [ ]:
from trainer import get_datas, QuantumReservoirNet, train

import torch.nn as nn
import torch
import torch.optim as optim

# nb_classes = 4
anomaly_ratio = 1

x_train, y_train, x_test, y_test = get_datas(400, anomaly_ratio)

# Convert to PyTorch tensors
x_train_t = torch.tensor(x_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

x_test_t = torch.tensor(x_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# Define the model, loss function and optimizer
model = QuantumReservoirNet()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 100
batch_size = 16

model = train(
    model,
    x_train_t,
    y_train_t,
    x_test_t,
    y_test_t,
    criterion,
    optimizer,
    epochs=epochs,
    batch_size=batch_size,
)

Epoch 1/100 | Loss: 1.3693 | Train acc: 0.250 | Test acc: 0.250
Epoch 2/100 | Loss: 1.3874 | Train acc: 0.250 | Test acc: 0.250
Epoch 3/100 | Loss: 1.3916 | Train acc: 0.250 | Test acc: 0.250
Epoch 4/100 | Loss: 1.3588 | Train acc: 0.387 | Test acc: 0.370
Epoch 5/100 | Loss: 1.2301 | Train acc: 0.472 | Test acc: 0.477
Epoch 6/100 | Loss: 1.2562 | Train acc: 0.470 | Test acc: 0.470
Epoch 7/100 | Loss: 1.0684 | Train acc: 0.495 | Test acc: 0.465
Epoch 8/100 | Loss: 1.1341 | Train acc: 0.500 | Test acc: 0.477
Epoch 9/100 | Loss: 1.0882 | Train acc: 0.458 | Test acc: 0.433
Epoch 10/100 | Loss: 0.9126 | Train acc: 0.500 | Test acc: 0.482
Epoch 11/100 | Loss: 0.8374 | Train acc: 0.502 | Test acc: 0.482
Epoch 12/100 | Loss: 0.8467 | Train acc: 0.505 | Test acc: 0.482
Epoch 13/100 | Loss: 0.8368 | Train acc: 0.533 | Test acc: 0.490
Epoch 14/100 | Loss: 0.8150 | Train acc: 0.515 | Test acc: 0.477
Epoch 15/100 | Loss: 0.8729 | Train acc: 0.525 | Test acc: 0.502
Epoch 16/100 | Loss: 0.8889 | Trai

### The full pipeline and the role of quantum in it
The full pipeline can be understood by thinking of a satellite image of size 32x32 coming as an input to the algorithm. 

The first step is to process the image into 512 characteristic features of the image, using Resnet 18, a pretrained model available. We then use a PCA 10 algorithm to reduce to the 10 most important features. Then, we use these in a QSVM, algorithm, marking the first use of a quantum computer. This QSVM is only trained to say wheter or not an image is flagged as normal, we call it a QSVM 1 class. Thus, we have no idea of which class we obtained. We then need to use another algorithm, which is only applied if the image is flagged as anormal by the QSVM.

The same entry image is thus converted into 10 new features using a classical neural network. These 10 features are used in a photonic circuit that performs a boson sampling method to classify the data. Then, the algorithm as a return value of either `"not_flagged"`, `"airport"`, `"harbor"`, `"ship"`, or `"airplane"`.

Quantum is used as a machine learning engine. All of these task could be done via classical algorithms...

<img src="./pipeline.png" alt="Alt Text" width="1000">

Now, we can use our trained model to try to classify the true positives obtained in the first step. 

In [ ]:
anomaly_idx = np.where(preds == 1)[0]
class_preds = np.full(len(preds), -1, dtype=int)

if len(anomaly_idx) == 0:
    print("No flagged anomalies in the test set.")
else:
    x_flagged = x_test[anomaly_idx]

    if not isinstance(x_flagged, torch.Tensor):
        x_flagged = torch.tensor(x_flagged, dtype=torch.float32)

    model.eval()
    with torch.no_grad():
        logits_flagged = model(x_flagged)
        flagged_preds = logits_flagged.argmax(dim=1).cpu().numpy()

    class_preds[anomaly_idx] = flagged_preds

    print(f"Classified {len(anomaly_idx)} flagged samples through the second model.")
    for idx, pred in zip(anomaly_idx, flagged_preds):
        print(f"Sample {idx:03d} | predicted class = {pred}")

In [ ]:
# Ensure numpy
y_test = np.array(y_test)
preds = np.array(preds)
class_preds = np.array(class_preds)
y_class_test = np.array(class_preds)

# ====================================================
# 1. TRUE ANOMALIES DETECTED
# ====================================================

true_anomaly_idx = np.where((preds == 1) & (y_test == 1))[0]

print("\n==============================")
print("TRUE ANOMALIES DETECTED")
print("==============================")
print(f"Total detected true anomalies: {len(true_anomaly_idx)}\n")

for i in true_anomaly_idx:
    print(
        f"[TP] Sample {i:03d} | "
        f"Pred Class = {class_preds[i]} | "
        f"True Class = {y_class_test[i]}"
    )

# ====================================================
# 2. MISSED ANOMALIES
# ====================================================

missed_anomaly_idx = np.where((preds == 0) & (y_test == 1))[0]

print("\n==============================")
print("MISSED ANOMALIES (FN)")
print("==============================")
print(f"Total missed anomalies: {len(missed_anomaly_idx)}\n")

for i in missed_anomaly_idx:
    print(
        f"[FN] Sample {i:03d} | "
        f"Pred Class = {class_preds[i]} | "
        f"True Class = {y_class_test[i]}"
    )

# ====================================================
# 3. FALSE POSITIVES (optional but useful)
# ====================================================

fp_idx = np.where((preds == 1) & (y_test == 0))[0]

print("\n==============================")
print("FALSE POSITIVES")
print("==============================")
print(f"Total false positives: {len(fp_idx)}\n")

for i in fp_idx:
    print(
        f"[FP] Sample {i:03d} | "
        f"Pred Class = {class_preds[i]} | "
        f"(True class irrelevant - normal sample)"
    )

# ====================================================
# 4. CLASSIFICATION PERFORMANCE ON TRUE ANOMALIES ONLY
# ====================================================

if len(true_anomaly_idx) > 0:
    acc_on_anomalies = np.mean(
        class_preds[true_anomaly_idx] == y_class_test[true_anomaly_idx]
    )

    print("\n==============================")
    print("CLASSIFICATION ON TRUE ANOMALIES")
    print("==============================")
    print(f"Accuracy on anomalies: {acc_on_anomalies:.4f}")
else:
    print("\nNo true anomalies detected → cannot compute anomaly-class accuracy")